# HarmonyRL — training run (Kaggle)

Supervised pretraining on MAESTRO, then PPO fine-tuning.

**Before you run anything, in the right-hand panel:**

1. **Accelerator** -> `GPU T4 x2` (or P100).
2. **Internet** -> `On`. The notebook clones the repo and downloads MAESTRO.

Kaggle gives ~30 GPU hours/week and a 12h session cap. Stage 1 takes 5-8h on a T4,
so save the checkpoint as a Kaggle Dataset before the session ends (last cell).


## 0. Check the GPU


In [ ]:
import sys, torch

assert torch.cuda.is_available(), (
    'No GPU. Set Accelerator to GPU in the right-hand panel and restart.'
)
print('torch     ', torch.__version__)
print('gpu       ', torch.cuda.get_device_name(0))
print('vram (GB) ', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print('python    ', sys.version.split()[0])


## 1. Get the code


In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/SupratikB23/HarmonyRL.git'
BRANCH   = 'main'   # change if you pushed the rewrite to a branch
REPO     = Path('/kaggle/working/HarmonyRL')

if not REPO.exists():
    !git clone --depth 1 -b $BRANCH $REPO_URL $REPO
else:
    !cd $REPO && git pull

os.chdir(REPO)
!pip install -q -r requirements.txt
print('cwd:', os.getcwd())


## 2. Get MAESTRO

Pulled straight from Magenta — nothing to upload. ~58 MB zip, 1276 MIDI files.

*Alternative:* add the MAESTRO dataset via **+ Add Input** and point `data.root` at
`/kaggle/input/...` instead. Downloading is simpler and needs no dataset search.


In [ ]:
from pathlib import Path

MAESTRO_URL = ('https://storage.googleapis.com/magentadata/datasets/'
               'maestro/v3.0.0/maestro-v3.0.0-midi.zip')
DATA = Path('data/maestro')

if not DATA.exists() or not any(DATA.rglob('*.mid*')):
    DATA.mkdir(parents=True, exist_ok=True)
    !curl -L -o maestro.zip $MAESTRO_URL
    !unzip -q -o maestro.zip -d $DATA
    !rm -f maestro.zip

n_midi = len(list(DATA.rglob('*.mid*')))
print(f'{n_midi} MIDI files under {DATA}')
assert n_midi > 1000, 'expected ~1276 files; check the download'


## 3. Smoke test

Run the suite before spending GPU hours. If it fails, stop and fix it.


In [ ]:
!python -m pytest tests -q


## 4. Supervised pretraining

~25M params over ~41M tokens. Watch **val ppl** — it should fall well under 10.

The first run tokenizes all 1276 files into `.cache/` (a few minutes); later runs
reuse it. If you hit OOM on a 16GB card, drop `batch_size` to 8 and `max_seq_len`
to 512 in the config.


In [ ]:
!python scripts/train_supervised.py --config notebooks/configs/supervised_gpu.yaml


## 5. PPO fine-tuning

Starts from the supervised checkpoint and keeps a frozen copy as the reference policy.

**Watch two numbers together:**

- `R` rising is only good if `diversity` stays near 1.0.
- `R` up while `diversity` falls = the policy is hacking the reward. Kill it, raise
  `kl_coef` or the `diversity` weight.
- `kl` should drift up slowly; a sudden jump means it is running from the reference.


In [ ]:
!python scripts/train_rl.py --config notebooks/configs/rl_gpu.yaml


## 6. Generate and inspect

`max_repeat_run` is the honest check: low single digits is healthy, a large number
means the model is stuck on a note no matter what the reward says.


In [ ]:
!python scripts/infer.py --n_samples 6 --max_new_tokens 1024 \
    --output_dir outputs --no_audio


In [ ]:
import glob
import pretty_midi

for p in sorted(glob.glob('outputs/*.mid')):
    pm = pretty_midi.PrettyMIDI(p)
    notes = pm.instruments[0].notes if pm.instruments else []
    print(f'{p:28s} {len(notes):5d} notes  {pm.get_end_time():6.1f}s')


## 7. Keep your checkpoints

`/kaggle/working` is wiped when the session ends. Either download the files from the
**Output** tab, or save them as a Kaggle Dataset so the next session can load them
as an input.


In [ ]:
!ls -lh checkpoints outputs
print('
Download checkpoints/*.pt from the Output tab before the session ends.')
